# 🧠 Character-Level Language Models with Recurrent Neural Networks

### A complete walkthrough — from the single neuron to Shakespeare

> *Based on Andrej Karpathy — "The Unreasonable Effectiveness of Recurrent Neural Networks" (2015)*  
> https://karpathy.github.io/2015/05/21/rnn-effectiveness/

---

This notebook converts the handout into **live, runnable code**.
Every concept is explained, every equation is implemented, and every result is visualised.

| # | Topic | Key idea |
|---|-------|----------|
| 1 | Setup & vocabulary | Encoding characters as numbers |
| 2 | One-hot encoding | From characters to vectors |
| 3 | RNN architecture | The three weight matrices |
| 4 | Forward pass | Step-by-step through `"hello"` |
| 5 | Softmax + Loss | Cross-entropy explained |
| 6 | BPTT | Backprop through time + vanishing gradients |
| 7 | Full training loop | NumPy from scratch |
| 8 | Text generation | Autoregressive sampling + temperature |
| 9 | PyTorch LSTM | Real training on Shakespeare |
| 10 | Experiments | Temperature, embeddings, frequency analysis |

**Runtime:** GPU is optional but speeds up Section 9. Go to `Runtime → Change runtime type → T4 GPU`.

---
## ⚙️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch, torch.nn as nn, torch.optim as optim
import math, random, requests
from collections import Counter
from sklearn.decomposition import PCA

np.random.seed(42)
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print('All imports OK ✓')

---
## 📌 Section 1 — The Setup: Vocabulary and Training Sequence

### What does "character-level" mean?

A **character-level language model** predicts the **next character** given all characters seen so far.
It has no built-in knowledge of words, grammar, or syntax — it learns everything purely from
statistical patterns of which characters follow which.

Karpathy's minimal working example:
- **Vocabulary**: `{h, e, l, o}` — just 4 unique characters
- **Training word**: `"hello"` — 5 characters

### The four training examples hidden in `"hello"`

One word contains **four independent prediction tasks**:

| Step | Context seen | Must predict | Task |
|------|-------------|--------------|------|
| t=1 | `"h"` | `"e"` | P(e \| h) |
| t=2 | `"h", "e"` | `"l"` | P(l \| h,e) |
| t=3 | `"h", "e", "l"` | `"l"` | P(l \| h,e,l) |
| t=4 | `"h", "e", "l", "l"` | `"o"` | P(o \| h,e,l,l) |

> ⚠️ **Key insight**: Steps t=3 and t=4 receive the **identical input** `"l"` but require
> **different outputs** (`"l"` vs `"o"`). The model must use its hidden state — its memory
> of the context — to distinguish them. This is exactly what the recurrent connection does.

In [ ]:
# ─── Vocabulary and encoding ───────────────────────────────────
VOCAB      = ['h', 'e', 'l', 'o']
VOCAB_SIZE = len(VOCAB)
char_to_idx = {ch: i for i, ch in enumerate(VOCAB)}
idx_to_char = {i: ch for i, ch in enumerate(VOCAB)}

TRAIN_SEQ = 'hello'

print('Vocabulary:', VOCAB)
print('char → idx:', char_to_idx)
print()
print('Four training examples extracted from "hello":')
header = f'  {"Step":<6} {"Input":<14} {"Target":<10} {"Task"}'
print(header)
print('  ' + '-'*50)
for t in range(len(TRAIN_SEQ)-1):
    ctx = TRAIN_SEQ[:t+1]
    tgt = TRAIN_SEQ[t+1]
    task = f'P({tgt} | {ctx})'
    print(f'  t={t+1}    "{ctx}"          "{tgt}"         {task}')

---
## 📌 Section 2 — Input Encoding: One-Hot Vectors

The RNN needs **numerical vectors**, not raw characters. We use **one-hot encoding**
(Karpathy calls it *1-of-k encoding*): a vector of zeros with a **single 1** at
the character's vocabulary index.

```
Vocabulary index:   h=0   e=1   l=2   o=3

  "h"  →  [1, 0, 0, 0]
  "e"  →  [0, 1, 0, 0]
  "l"  →  [0, 0, 1, 0]
  "o"  →  [0, 0, 0, 1]
```

With vocabulary size V=4, each input vector `x_t` has shape **(4,)**.
This scales naturally: a 65-character vocabulary gives 65-dimensional vectors.

In [ ]:
def one_hot(char):
    """Convert a single character to a one-hot numpy vector."""
    vec = np.zeros(VOCAB_SIZE)
    vec[char_to_idx[char]] = 1.0
    return vec

# Show all four one-hot vectors
print('One-hot encoding for each character:')
print(f'  {"char":<6} {"vector":<24} {"active index"}')
print('  ' + '-'*42)
for ch in VOCAB:
    v = one_hot(ch)
    vec_s = '[' + ', '.join(str(int(x)) for x in v) + ']'
    print(f'  "{ch}"    {vec_s:<24} → index {char_to_idx[ch]}')

# Encode the full sequence
inputs  = [one_hot(c) for c in TRAIN_SEQ[:-1]]   # x vectors: h, e, l, l
targets = [char_to_idx[c] for c in TRAIN_SEQ[1:]] # target indices: e, l, l, o
print()
print(f'inputs  → list of {len(inputs)} vectors, each shape ({VOCAB_SIZE},)')
print(f'targets → {targets}  (e=1, l=2, l=2, o=3)')

In [ ]:
# ── Visualise one-hot encoding ───────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.2))
matrix = np.array([one_hot(c) for c in VOCAB])

im = ax.imshow(matrix, cmap='Purples', vmin=0, vmax=1.3, aspect='auto')
ax.set_xticks(range(VOCAB_SIZE))
ax.set_xticklabels([f'dim {i}' for i in range(VOCAB_SIZE)], fontsize=11)
ax.set_yticks(range(VOCAB_SIZE))
ax.set_yticklabels([f'"{c}"' for c in VOCAB], fontsize=13, fontstyle='italic')
ax.set_title('One-Hot Encoding  —  vocabulary {h, e, l, o}', fontsize=13, pad=10)
for i in range(VOCAB_SIZE):
    for j in range(VOCAB_SIZE):
        v = int(matrix[i, j])
        ax.text(j, i, str(v), ha='center', va='center',
                fontsize=16, fontweight='bold', color='white' if v else '#999')
plt.tight_layout(); plt.show()
print('Purple cell = the active ("hot") position for each character.')

---
## 📌 Section 3 — RNN Architecture: The Three Weight Matrices

An RNN applies the **same computation** at every time step:

$$h_t = \tanh\!\bigl(\underbrace{W_{xh}}_{\text{input→hidden}} \cdot x_t \;+\;
\underbrace{W_{hh}}_{\text{hidden→hidden}} \cdot h_{t-1} \;+\; b_h\bigr)$$

$$y_t = W_{hy} \cdot h_t + b_y$$

| Matrix | Shape | Role |
|--------|-------|------|
| $W_{xh}$ | `(H × V)` | **Input→hidden**: encodes what the current character means |
| $W_{hh}$ | `(H × H)` | **Hidden→hidden**: the **recurrent connection** — carries memory |
| $W_{hy}$ | `(V × H)` | **Hidden→output**: reads a prediction from the hidden state |

The **same three matrices** are used at every time step (*weight sharing across time*).
Parameter count is fixed regardless of sequence length.

This is Karpathy's exact Python implementation from his blog:

In [ ]:
class VanillaRNN:
    """
    Minimal character-level RNN.
    Faithful implementation of Karpathy's blog pseudocode.
    """
    def __init__(self, input_size, hidden_size, output_size, seed=0):
        np.random.seed(seed)
        s = 0.01  # small random init
        self.Wxh = np.random.randn(hidden_size, input_size)  * s  # input→hidden
        self.Whh = np.random.randn(hidden_size, hidden_size) * s  # hidden→hidden (recurrent!)
        self.Why = np.random.randn(output_size, hidden_size) * s  # hidden→output
        self.bh  = np.zeros((hidden_size, 1))                      # hidden bias
        self.by  = np.zeros((output_size, 1))                      # output bias
        self.h   = np.zeros((hidden_size, 1))                      # hidden state (memory)
        self.H   = hidden_size

    def step(self, x):
        """Forward pass — one time step. x: (vocab_size, 1)."""
        # Mix past memory + new input, squash with tanh
        self.h = np.tanh(self.Wxh @ x + self.Whh @ self.h + self.bh)
        # Read prediction from updated hidden state
        y = self.Why @ self.h + self.by
        return y, self.h.copy()

    def reset(self):
        self.h = np.zeros((self.H, 1))


# Instantiate with Karpathy's figure dimensions (hidden=3, vocab=4)
HIDDEN = 3
rnn = VanillaRNN(VOCAB_SIZE, HIDDEN, VOCAB_SIZE)

print('VanillaRNN created:')
for name, mat in [('Wxh', rnn.Wxh), ('Whh', rnn.Whh), ('Why', rnn.Why),
                   ('bh',  rnn.bh),  ('by',  rnn.by)]:
    print(f'  {name:4s}  shape: {str(mat.shape):<14}',
          '← recurrent connection!' if name == 'Whh' else '')
total = sum(m.size for m in [rnn.Wxh, rnn.Whh, rnn.Why, rnn.bh, rnn.by])
print(f'  Total parameters: {total}')

---
## 📌 Section 4 — The Forward Pass: Step by Step Through `"hell"`

Let's trace the forward pass exactly, watching the hidden state update at each step.
The network is **untrained** (random weights) so scores will be poor — that's fine.
The structure of the computation is what matters.

In [ ]:
# ── Trace the complete forward pass ──────────────────────────
rnn.reset()
h_history = []   # hidden states at each step
y_history = []   # output logits at each step

print('='*68)
print('FORWARD PASS: input sequence "hell", predicting next character')
print('='*68)

for t, ch in enumerate(TRAIN_SEQ[:-1]):   # h, e, l, l
    x      = one_hot(ch).reshape(-1, 1)   # (4,1) one-hot vector
    tgt_ch = TRAIN_SEQ[t+1]               # correct next character
    tgt_i  = char_to_idx[tgt_ch]

    y, h = rnn.step(x)
    h_history.append(h.flatten().copy())
    y_history.append(y.flatten().copy())

    print(f'\n  t={t+1}  input="{ch}"  →  target="{tgt_ch}"')
    print(f'  z  = Wxh·x + Whh·h_prev + bh')
    print(f'  h{t+1} = tanh(z) = [{"  ".join(f"{v:+.3f}" for v in h.flatten())}]')
    print(f'  y{t+1} output scores (logits):')
    for vi, vc in enumerate(VOCAB):
        mark = '  ← TARGET ✓ (want this HIGH)' if vi == tgt_i else ''
        icon = '▶' if vi == tgt_i else ' '
        print(f'    {icon} "{vc}": {y.flatten()[vi]:+.4f}{mark}')

print()
print('='*68)
print('Observe: t=3 and t=4 have IDENTICAL input "l" but the hidden')
print('states h3 and h4 are DIFFERENT — the recurrent connection Whh')
print('makes sure memory of "hel" vs "hell" context is preserved.')
print('='*68)

In [ ]:
# ── Visualise: hidden states + output scores side by side ───
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
step_labels = ['t=1\n"h"→"e"','t=2\n"e"→"l"','t=3\n"l"→"l"','t=4\n"l"→"o"']
targets_idx = [char_to_idx[c] for c in TRAIN_SEQ[1:]]

# Hidden state trajectories
ax = axes[0]
h_arr = np.array(h_history)
for ni in range(HIDDEN):
    ax.plot(range(1,5), h_arr[:,ni], 'o-', lw=2, ms=8, label=f'neuron {ni}')
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xticks(range(1,5)); ax.set_xticklabels(step_labels, fontsize=9)
ax.set_title('Hidden state $h_t$ (3 neurons)\nThe network\'s "memory"', fontsize=11)
ax.set_ylabel('Activation (tanh output)'); ax.set_ylim(-1.1,1.1)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Output score heatmap
ax = axes[1]
y_arr = np.array(y_history).T   # (vocab, steps)
im = ax.imshow(y_arr, cmap='RdYlGn', aspect='auto', vmin=-0.2, vmax=0.2)
ax.set_xticks(range(4)); ax.set_xticklabels(step_labels, fontsize=9)
ax.set_yticks(range(VOCAB_SIZE))
ax.set_yticklabels([f'"{c}"' for c in VOCAB], fontsize=12, fontstyle='italic')
ax.set_title('Output logits $y_t$\nBoxes = correct target', fontsize=11)
for t,ti in enumerate(targets_idx):
    ax.add_patch(plt.Rectangle((t-.5,ti-.5),1,1,fill=False,ec='navy',lw=2.5))
for t in range(4):
    for v in range(VOCAB_SIZE):
        ax.text(t,v,f'{y_arr[v,t]:.2f}',ha='center',va='center',fontsize=8)
plt.colorbar(im, ax=ax)

# Hidden state heatmap — highlight identical inputs with different memory
ax = axes[2]
im2 = ax.imshow(h_arr.T, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(step_labels, fontsize=9)
ax.set_yticks(range(HIDDEN))
ax.set_yticklabels([f'h[{i}]' for i in range(HIDDEN)])
ax.set_title('Hidden state heatmap\nSame input "l", different h_t!', fontsize=11)
# Highlight the two identical 'l' inputs
for t in [2,3]:
    ax.add_patch(plt.Rectangle((t-.5,-.5),1,HIDDEN,
                 facecolor='yellow',alpha=0.18,ec='goldenrod',lw=2))
for t in range(4):
    for n in range(HIDDEN):
        ax.text(t,n,f'{h_arr[t,n]:.2f}',ha='center',va='center',fontsize=9)
plt.colorbar(im2, ax=ax)
ax.text(2.5,-0.9,'← both "l", different h_t',ha='center',fontsize=8,color='goldenrod')

plt.suptitle('RNN Forward Pass through "hell"', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
## 📌 Section 5 — Softmax and Cross-Entropy Loss

### Softmax — converting raw scores to probabilities

The output $y_t$ is a vector of raw **logit scores**. To get probabilities we apply **Softmax**:

$$p_t = \text{softmax}(y_t) \qquad p_t[i] = \frac{e^{y_t[i]}}{\sum_j e^{y_t[j]}}$$

All values become positive and sum to 1.

### Cross-entropy loss — how wrong are we?

$$L_t = -\log\bigl(p_t[\text{correct char}]\bigr) \qquad L = \sum_{t=1}^{T} L_t$$

- Perfect prediction → $p = 1.0$ → $L = -\log(1) = 0$  
- Random (uniform over 4 chars) → $p = 0.25$ → $L = -\log(0.25) \approx 1.39$  
- Completely wrong → $p \approx 0$ → $L \to \infty$

In [ ]:
def softmax(x):
    """Numerically stable softmax."""
    e = np.exp(x - np.max(x))
    return e / e.sum()

def cross_entropy(probs, target_idx):
    return -np.log(probs[target_idx] + 1e-8)


# ── Compute softmax + loss at each step ────────────────────
rnn.reset()
total_loss = 0.0

print('='*70)
print('SOFTMAX + CROSS-ENTROPY LOSS at each time step')
print('='*70)

for t, ch in enumerate(TRAIN_SEQ[:-1]):
    x      = one_hot(ch).reshape(-1,1)
    tgt_ch = TRAIN_SEQ[t+1]
    tgt_i  = char_to_idx[tgt_ch]

    y, _  = rnn.step(x)
    probs = softmax(y.flatten())
    loss  = cross_entropy(probs, tgt_i)
    total_loss += loss

    print(f'\n  t={t+1}: input="{ch}" → target="{tgt_ch}"')
    print(f'  Logits y: {np.round(y.flatten(),3)}')
    print(f'  Probs  p: ', end='')
    for vi,(vc,p) in enumerate(zip(VOCAB, probs)):
        mark = ' ←✓' if vi==tgt_i else '   '
        print(f'"{vc}":{p:.3f}{mark}', end='  ')
    print()
    print(f'  Loss:  −log({probs[tgt_i]:.4f}) = {loss:.4f}')

rand_baseline = -np.log(1/VOCAB_SIZE)*4
print(f'\nTotal loss:        {total_loss:.4f}')
print(f'Random baseline:   {rand_baseline:.4f}  (uniform over {VOCAB_SIZE} chars × {len(TRAIN_SEQ)-1} steps)')
print(f'\nGoal of training: push total loss toward 0.0')

In [ ]:
# ── Visualise softmax and loss ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Softmax: how score → probability
ax = axes[0]
logit_range = np.linspace(-4, 4, 200)
p_of_target = [softmax(np.array([l, 0, 0, 0]))[0] for l in logit_range]
ax.plot(logit_range, p_of_target, 'steelblue', lw=2.5)
ax.axhline(0.25, color='red', ls='--', alpha=0.7, label='Random baseline (0.25)')
ax.axvline(0, color='gray', ls=':', alpha=0.5)
ax.fill_between(logit_range, p_of_target, alpha=0.1, color='steelblue')
ax.set_xlabel('Logit score for target character', fontsize=11)
ax.set_ylabel('Softmax probability of target', fontsize=11)
ax.set_title('How output score maps to probability\n(other chars fixed at 0)', fontsize=11)
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1)

# Cross-entropy: how probability → loss
ax = axes[1]
p_range = np.linspace(0.01, 1.0, 200)
ax.plot(p_range, -np.log(p_range), 'coral', lw=2.5)
ax.scatter([0.25], [-np.log(0.25)], s=100, color='red', zorder=5,
           label=f'Random (p=0.25, loss={-np.log(0.25):.2f})')
ax.scatter([1.0], [0.0], s=100, color='green', zorder=5,
           label='Perfect (p=1.0, loss=0.0)')
ax.set_xlabel('Probability assigned to correct character', fontsize=11)
ax.set_ylabel('Cross-entropy loss  −log(p)', fontsize=11)
ax.set_title('Cross-entropy loss\n(lower = better prediction)', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(0, 5)

plt.suptitle('Softmax → Probability → Loss', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 📌 Section 6 — Backpropagation Through Time (BPTT)

To train the RNN we need $\partial L / \partial W$ for each weight matrix.
BPTT applies the **chain rule** through the unrolled network:

$$\frac{\partial L}{\partial W_{hh}} = \sum_t \frac{\partial L_t}{\partial W_{hh}}$$

At step $t=3$, the gradient must flow **backward through all previous steps**:

$$\frac{\partial L_3}{\partial W_{hh}} = \frac{\partial L_3}{\partial y_3}
\cdot \frac{\partial y_3}{\partial h_3}
\cdot \underbrace{\frac{\partial h_3}{\partial h_2} \cdot \frac{\partial h_2}{\partial h_1}}
_{\text{flows backward in time}}
\cdot \frac{\partial h_1}{\partial W_{hh}}$$

Each hidden-to-hidden Jacobian is:
$$\frac{\partial h_t}{\partial h_{t-1}} = \tanh'(z_t) \cdot W_{hh} = (1 - h_t^2) \cdot W_{hh}$$

Since $|\tanh'| \leq 1$, multiplying $T$ such terms → **gradients vanish exponentially**.
The network stops learning from tokens more than ~10 steps in the past.

> 💡 **LSTM solution**: Replace the multiplicative update with an **additive** one:
> $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$
> so $\partial C_t / \partial C_{t-1} = f_t$ — a learned value, trainable to stay near 1.

In [ ]:
def bptt_forward_backward(rnn, inputs_list, targets_list):
    """
    Full forward pass + BPTT backward pass.
    Returns loss and gradient dict.
    Based on Karpathy's minimal char-rnn implementation.
    """
    T = len(inputs_list)
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = rnn.h.copy()   # initial hidden state h0

    # ── Forward pass ─────────────────────────────────────────
    loss = 0.0
    for t in range(T):
        xs[t] = inputs_list[t].reshape(-1,1)
        hs[t] = np.tanh(rnn.Wxh @ xs[t] + rnn.Whh @ hs[t-1] + rnn.bh)
        ys[t] = rnn.Why @ hs[t] + rnn.by
        ps[t] = softmax(ys[t].flatten()).reshape(-1,1)
        loss += -np.log(ps[t][targets_list[t],0] + 1e-8)

    # ── Backward pass (BPTT) ──────────────────────────────────
    dWxh = np.zeros_like(rnn.Wxh)
    dWhh = np.zeros_like(rnn.Whh)
    dWhy = np.zeros_like(rnn.Why)
    dbh  = np.zeros_like(rnn.bh)
    dby  = np.zeros_like(rnn.by)
    dh_next = np.zeros_like(hs[0])   # gradient from future steps

    for t in reversed(range(T)):    # go backward: T-1, T-2, ... 0
        # Gradient of cross-entropy + softmax
        dy = ps[t].copy()
        dy[targets_list[t]] -= 1.0   # -= 1 at correct class

        # Gradients for output layer
        dWhy += dy @ hs[t].T
        dby  += dy

        # Gradient flowing back to hidden state (output + future steps)
        dh = rnn.Why.T @ dy + dh_next

        # ⚠ Gradient through tanh: (1 - h²)  ← source of vanishing gradients
        dtanh = (1 - hs[t]**2) * dh

        # Gradients for input & recurrent weights
        dWxh += dtanh @ xs[t].T
        dWhh += dtanh @ hs[t-1].T
        dbh  += dtanh

        # Pass gradient to previous time step via Whh
        dh_next = rnn.Whh.T @ dtanh

    # Gradient clipping — prevents exploding gradients
    for g in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(g, -5, 5, out=g)

    rnn.h = hs[T-1]   # carry hidden state forward
    return loss, dict(Wxh=dWxh, Whh=dWhh, Why=dWhy, bh=dbh, by=dby)

print('BPTT function defined ✓')

In [ ]:
# ── Demonstrate vanishing gradients numerically ─────────────
print('Vanishing gradient demo: how gradient shrinks per time step')
print()
print(f'  Each step multiplies gradient by tanh\'(z) = (1 − h²)')
print(f'  Since |tanh\'| ≤ 1, multiplying T times shrinks it:')
print()
print(f'  {"h_t value":<14} {"tanh\' value":<14} {"After 10 steps":<18} {"After 20 steps"}')
print(f'  {"-"*65}')
for h_val in [0.3, 0.6, 0.8, 0.95]:
    g = 1 - h_val**2
    print(f'  {h_val:<14.2f} {g:<14.4f} {g**10:<18.8f} {g**20:.12f}')
print()
print('→ High hidden activations → near-zero tanh\' → gradient vanishes fast')
print('→ Model cannot learn long-range dependencies (> ~10 steps)')
print('→ LSTM solves this with an ADDITIVE cell state update')

In [ ]:
# ── Visualise vanishing gradients ────────────────────────────
fig, axes = plt.subplots(1,2, figsize=(13,4))

steps = np.arange(1,31)

# Panel 1: gradient magnitude vs steps for different h values
ax = axes[0]
configs = [('h=0.3',0.3,'green'), ('h=0.6',0.6,'orange'),
           ('h=0.8',0.8,'red'),   ('h=0.95',0.95,'darkred')]
for label,h,col in configs:
    g = 1-h**2
    ax.semilogy(steps, [g**t for t in steps], 'o-',
               ms=4, lw=2, color=col, label=f'{label}  (tanh\'={g:.2f})')
ax.axhline(1e-3, ls='--', color='gray', alpha=0.6, label='≈ zero (1e-3)')
ax.set_xlabel('Steps backward in time'); ax.set_ylabel('Gradient magnitude (log scale)')
ax.set_title('Vanilla RNN: Vanishing Gradients', fontsize=12)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 2: RNN vs LSTM comparison
ax = axes[1]
rnn_g  = [0.36**t for t in steps]
lstm_g = [min(1, 0.97**t + 0.02) for t in steps]   # approximate LSTM
ax.semilogy(steps, rnn_g,  'r-o', ms=4, lw=2, label='Vanilla RNN')
ax.semilogy(steps, lstm_g, 'g-o', ms=4, lw=2, label='LSTM (approx.)')
ax.axhline(1e-2, ls='--', color='gray', alpha=0.5)
ax.set_xlabel('Steps backward in time'); ax.set_ylabel('Gradient magnitude (log scale)')
ax.set_title('RNN vs LSTM: Gradient Flow Comparison', fontsize=12)
ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.suptitle('The Vanishing Gradient Problem', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 📌 Section 7 — Full Training Loop (NumPy from Scratch)

Now we put everything together. We train on `"hello"` using **Adagrad** —
a per-parameter adaptive learning rate method.
Karpathy uses RMSProp or Adam; Adagrad is conceptually similar and simpler to implement.

The update rule for Adagrad:
$$W \leftarrow W - \frac{\eta}{\sqrt{m + \epsilon}} \cdot g \qquad m \mathrel{+}= g^2$$

where $m$ accumulates the squared gradients, automatically reducing the effective
learning rate for frequently-updated parameters.

In [ ]:
def train(hidden_size=12, n_epochs=800, lr=0.1, seq=TRAIN_SEQ, verbose=True):
    rnn  = VanillaRNN(VOCAB_SIZE, hidden_size, VOCAB_SIZE)
    mem  = {k: np.zeros_like(getattr(rnn,k))
            for k in ['Wxh','Whh','Why','bh','by']}   # Adagrad memory

    inps = [one_hot(c) for c in seq[:-1]]
    tgts = [char_to_idx[c] for c in seq[1:]]
    history = []

    for epoch in range(n_epochs):
        rnn.reset()
        loss, grads = bptt_forward_backward(rnn, inps, tgts)
        history.append(loss)

        # Adagrad weight update
        for key, grad in grads.items():
            mem[key] += grad**2
            param      = getattr(rnn, key)
            param     -= lr * grad / (np.sqrt(mem[key]) + 1e-8)

        if verbose and (epoch % 200 == 0 or epoch == n_epochs-1):
            print(f'  Epoch {epoch:4d}  loss: {loss:.4f}')

    return rnn, history

print('Training RNN on "hello" ...')
print('-'*35)
trained_rnn, loss_hist = train(hidden_size=12, n_epochs=800, lr=0.1)
rand_bl = -np.log(1/VOCAB_SIZE) * (len(TRAIN_SEQ)-1)
print()
print(f'Final loss:      {loss_hist[-1]:.4f}')
print(f'Random baseline: {rand_bl:.4f}')
print(f'Improvement:     {(1 - loss_hist[-1]/rand_bl)*100:.1f}%')

In [ ]:
# ── Training curve + learned predictions heatmap ─────────────
fig, axes = plt.subplots(1,2, figsize=(13,4))

# Loss curve
ax = axes[0]
ax.plot(loss_hist, 'steelblue', lw=2, label='Training loss')
ax.axhline(rand_bl, color='red', ls='--', alpha=0.7,
           label=f'Random baseline ({rand_bl:.2f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
ax.set_title('Training Loss on "hello"', fontsize=12)
ax.legend(); ax.grid(alpha=0.3)

# Predicted probabilities after training
ax = axes[1]
trained_rnn.reset()
prob_mat = []
for ch in TRAIN_SEQ[:-1]:
    x = one_hot(ch).reshape(-1,1)
    y,_ = trained_rnn.step(x)
    prob_mat.append(softmax(y.flatten()))
prob_mat = np.array(prob_mat).T  # (vocab, steps)

im = ax.imshow(prob_mat, cmap='Greens', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(4))
ax.set_xticklabels(['t=1\n"h"','t=2\n"e"','t=3\n"l"','t=4\n"l"'], fontsize=9)
ax.set_yticks(range(VOCAB_SIZE))
ax.set_yticklabels([f'"{c}"' for c in VOCAB], fontsize=12, fontstyle='italic')
ax.set_title('Predicted probabilities (after training)\nBoxes = correct next char', fontsize=11)
for t,ti in enumerate(char_to_idx[c] for c in TRAIN_SEQ[1:]):
    ax.add_patch(plt.Rectangle((t-.5,ti-.5),1,1,fill=False,ec='navy',lw=3))
for t in range(4):
    for v in range(VOCAB_SIZE):
        ax.text(t,v,f'{prob_mat[v,t]:.2f}',ha='center',va='center',fontsize=9)
plt.colorbar(im, ax=ax)

plt.suptitle('Training Results', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('The network has learned to assign highest probability to the correct next character.')

---
## 📌 Section 8 — Generating New Text: Autoregressive Sampling

Once trained, the RNN generates text by **feeding its own outputs back as inputs**.
This is called **autoregressive generation**:

```
1. Seed character → x_1 = one_hot(seed)
2. Forward pass → logits y_1
3. Softmax → probability distribution p_1
4. Sample next char from p_1
5. next_char → x_2  (feed back)  →  repeat
```

### Temperature

Before softmax, divide logits by temperature $T$: &ensp; $p = \text{softmax}(y / T)$

| Temperature | Effect |
|-------------|--------|
| $T < 1$ | Sharper distribution — more confident, more repetitive |
| $T = 1$ | Standard sampling |
| $T > 1$ | Flatter distribution — more diverse, more mistakes |

> Karpathy: *"Setting temperature very near zero will give the most likely thing that*
> *Paul Graham might say"* — and it loops infinitely about startups.

In [ ]:
def generate(rnn, seed, n_chars=60, temperature=1.0):
    """
    Generate text autoregressively from a trained VanillaRNN.
    seed:        starting character (must be in VOCAB)
    temperature: controls randomness (T<1=conservative, T>1=creative)
    """
    rnn.reset()
    x = one_hot(seed).reshape(-1,1)
    out = [seed]

    for _ in range(n_chars):
        y, _ = rnn.step(x)                              # forward pass
        p    = softmax(y.flatten() / temperature)       # apply temperature
        idx  = np.random.choice(VOCAB_SIZE, p=p)        # sample
        ch   = idx_to_char[idx]
        out.append(ch)
        x = one_hot(ch).reshape(-1,1)                   # feed back

    return ''.join(out)


print('='*65)
print('Text generation: seed="h", varying temperature')
print('='*65)
for T in [0.1, 0.5, 1.0, 1.5, 2.5]:
    samples = [generate(trained_rnn, 'h', n_chars=40, temperature=T)
               for _ in range(4)]
    print(f'  T={T:<4}:  {"  |  ".join(samples)}')

print()
print('Low T  → almost always outputs the most likely sequence (repeats "hello")')
print('High T → ignores the learned distribution, outputs random characters')

In [ ]:
# ── Visualise how temperature reshapes the distribution ─────
fig, axes = plt.subplots(1,3, figsize=(14,4))
trained_rnn.reset()
x = one_hot('h').reshape(-1,1)
y,_ = trained_rnn.step(x)

for ax, T, label in zip(axes,
                         [0.3, 1.0, 2.0],
                         ['T=0.3 (conservative)', 'T=1.0 (standard)', 'T=2.0 (creative)']):
    p = softmax(y.flatten()/T)
    bars = ax.bar(VOCAB, p, color=['#2E86C1','#27AE60','#E67E22','#C0392B'],
                  width=0.5, alpha=0.85)
    ax.set_ylim(0, 1)
    ax.set_title(label, fontsize=12)
    ax.set_ylabel('Probability' if ax==axes[0] else '')
    ax.set_xlabel('Next character')
    for bar, prob in zip(bars, p):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{prob:.3f}', ha='center', va='bottom', fontsize=10)
    ax.grid(alpha=0.3, axis='y')

plt.suptitle('Effect of Temperature on Output Distribution\n(after seeing "h", predicting next char)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 📌 Section 9 — PyTorch LSTM: Real Training on Shakespeare

Now we scale up to Karpathy's actual setup:
- **LSTM** instead of vanilla RNN (solves vanishing gradients)
- **Shakespeare** as the training corpus (~1M characters)
- **PyTorch** for GPU-accelerated training

### Why LSTM?

The LSTM replaces the single hidden state update with a **cell state** $C_t$
updated **additively** — this is the key insight:

| | Vanilla RNN | LSTM |
|---|---|---|
| Update type | Multiplicative: $h_t = \tanh(W \cdot h_{t-1} + \ldots)$ | Additive: $C_t = f_t C_{t-1} + i_t \tilde{C}_t$ |
| Gradient | $\partial h_t / \partial h_{t-1} = \tanh' \cdot W \leq 1$ | $\partial C_t / \partial C_{t-1} = f_t$ (learnable) |
| Long range | Vanishes after ~10 steps | Can learn 100s of steps |
| Parameters | 3 matrices | 4× more (4 gate matrices) |

In [ ]:
# ── Download Karpathy's Shakespeare dataset ──────────────────
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
print('Downloading Shakespeare...')
text = requests.get(URL).text
print(f'Downloaded: {len(text):,} characters')
print()
print('First 400 characters:')
print('-'*50)
print(text[:400])

In [ ]:
# ── Build vocabulary from Shakespeare ────────────────────────
chars     = sorted(set(text))
V         = len(chars)
stoi      = {ch:i for i,ch in enumerate(chars)}
itos      = {i:ch for i,ch in enumerate(chars)}

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n    = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

print(f'Vocabulary: {V} unique characters')
print(f'Chars: {chars[:30]}...')
print(f'Training:   {len(train_data):,} chars')
print(f'Validation: {len(val_data):,} chars')

In [ ]:
# ── LSTM Language Model ──────────────────────────────────────
class CharLSTM(nn.Module):
    """
    Two-layer LSTM character language model.
    embedding → LSTM (2 layers) → linear
    """
    def __init__(self, vocab_size, embed_dim, hidden, n_layers, dropout=0.3):
        super().__init__()
        self.hidden    = hidden
        self.n_layers  = n_layers
        self.embed     = nn.Embedding(vocab_size, embed_dim)
        self.lstm      = nn.LSTM(embed_dim, hidden, n_layers,
                                  batch_first=True, dropout=dropout)
        self.fc        = nn.Linear(hidden, vocab_size)
        self.drop      = nn.Dropout(dropout)

    def forward(self, x, hidden=None):
        emb            = self.drop(self.embed(x))     # (B, T, embed_dim)
        out, hidden    = self.lstm(emb, hidden)        # (B, T, hidden)
        logits         = self.fc(self.drop(out))       # (B, T, vocab_size)
        return logits, hidden

    def init_hidden(self, B):
        h = torch.zeros(self.n_layers, B, self.hidden).to(device)
        c = torch.zeros(self.n_layers, B, self.hidden).to(device)
        return h, c


# ── Hyperparameters ───────────────────────────────────────────
EMBED   = 64
HIDDEN  = 256
LAYERS  = 2
SEQ_LEN = 100
BATCH   = 64
LR      = 3e-3
EPOCHS  = 5    # ← increase to 20+ for much better results

model = CharLSTM(V, EMBED, HIDDEN, LAYERS).to(device)
opt   = optim.Adam(model.parameters(), lr=LR)
crit  = nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model.parameters())
print(f'{LAYERS}-layer LSTM  |  hidden={HIDDEN}  |  {n_params:,} parameters')
print(f'Device: {device}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────
def get_batch(data, seq_len, batch_size):
    ix = torch.randint(len(data)-seq_len-1, (batch_size,))
    x  = torch.stack([data[i:i+seq_len]   for i in ix]).to(device)
    y  = torch.stack([data[i+1:i+seq_len+1] for i in ix]).to(device)
    return x, y

STEPS = 200
train_losses, val_losses = [], []

print(f'Training {EPOCHS} epochs × {STEPS} steps ...')
print(f'(Set EPOCHS=20 for much better Shakespeare generation)')
print('-'*55)

for epoch in range(EPOCHS):
    model.train()
    eloss = 0.0
    for _ in range(STEPS):
        xb, yb = get_batch(train_data, SEQ_LEN, BATCH)
        opt.zero_grad()
        logits, _ = model(xb)
        loss = crit(logits.view(-1,V), yb.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        eloss += loss.item()

    model.eval()
    with torch.no_grad():
        xv,yv = get_batch(val_data, SEQ_LEN, BATCH)
        vl,_  = model(xv)
        vloss = crit(vl.view(-1,V), yv.view(-1)).item()

    tl = eloss/STEPS
    train_losses.append(tl)
    val_losses.append(vloss)
    print(f'  Epoch {epoch+1:2d}/{EPOCHS}  train={tl:.4f}  val={vloss:.4f}  ppl={math.exp(vloss):.1f}')

print('\nTraining complete ✓')

In [ ]:
# ── Generate Shakespeare-style text ──────────────────────────
@torch.no_grad()
def generate_lstm(model, seed, n=500, temp=0.8):
    model.eval()
    # Encode seed and warm up hidden state
    ids    = [stoi[c] for c in seed if c in stoi]
    x      = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    hidden = model.init_hidden(1)

    if x.shape[1] > 1:               # warm up on all but last char
        _, hidden = model(x[:,:-1], hidden)
        x = x[:,-1:]

    out = seed
    for _ in range(n):
        logits, hidden = model(x, hidden)               # (1,1,V)
        p = torch.softmax(logits[0,-1]/temp, dim=-1).cpu().numpy()
        nxt = np.random.choice(V, p=p)
        out += itos[nxt]
        x = torch.tensor([[nxt]], dtype=torch.long).to(device)
    return out

print('='*65)
print('GENERATED SHAKESPEARE  (temperature=0.8)')
print('='*65)
print(generate_lstm(model, 'ROMEO:\n', n=600, temp=0.8))

In [ ]:
# ── Training curves ──────────────────────────────────────────
fig, axes = plt.subplots(1,2, figsize=(13,4))

ax = axes[0]
ax.plot(range(1,EPOCHS+1), train_losses, 'b-o', lw=2, label='Train')
ax.plot(range(1,EPOCHS+1), val_losses,   'r-o', lw=2, label='Validation')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
ax.set_title('LSTM Training on Shakespeare', fontsize=12)
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ppls = [math.exp(l) for l in val_losses]
ax.plot(range(1,EPOCHS+1), ppls, 'g-o', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Perplexity')
ax.set_title('Validation Perplexity\n(lower = model less surprised)', fontsize=11)
ax.grid(alpha=0.3)

plt.suptitle('LSTM Training Results', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 📌 Section 10 — Experiments

### 10.1 Temperature experiment

Generate the same seed with different temperatures and compare.

In [ ]:
print('Temperature experiment — seed: "HAMLET:\\n"')
print('='*65)
for T in [0.3, 0.7, 1.0, 1.5]:
    sample = generate_lstm(model, 'HAMLET:\n', n=250, temp=T)
    print(f'\n  ── T={T} ──────────────────────────────────────────────')
    print(f'  {sample[:250]}')

### 10.2 Character embedding visualisation

The embedding layer maps each character to a learned vector.
Characters that appear in similar contexts (e.g. all lowercase letters)
should cluster together in the embedding space.

In [ ]:
# ── PCA of character embeddings ──────────────────────────────
embs   = model.embed.weight.detach().cpu().numpy()
pca    = PCA(n_components=2)
coords = pca.fit_transform(embs)

fig, ax = plt.subplots(figsize=(12,8))
ax.scatter(coords[:,0], coords[:,1], alpha=0.2, s=15, color='steelblue')

for ch in chars:
    idx = stoi[ch]
    x,y = coords[idx]
    if ch.isupper():   col='red'
    elif ch.islower(): col='green'
    elif ch.isdigit(): col='orange'
    else:              col='purple'
    disp = '\\n' if ch=='\n' else ('SPC' if ch==' ' else ch)
    ax.annotate(f'"{disp}"', (x,y), fontsize=7, color=col, ha='center')
    ax.scatter(x, y, s=40, color=col, zorder=4, alpha=0.7)

handles = [mpatches.Patch(color=c,label=l) for c,l in
           [('red','Uppercase'),('green','Lowercase'),
            ('orange','Digits'),('purple','Punctuation')]]
ax.legend(handles=handles, fontsize=10)
ax.set_title('Character Embeddings (PCA projection)\nSimilar chars cluster together', fontsize=12)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 10.3 Character frequency: training data vs generated text

A well-trained model should produce character frequencies that match the training data.

In [ ]:
# ── Character frequency comparison ───────────────────────────
gen_text = generate_lstm(model, 'KING LEAR:\n', n=8000, temp=0.8)

train_freq = Counter(text[:15000])
gen_freq   = Counter(gen_text)

plot_chars = [c for c in chars if c.isalpha() or c in '.,:;?!\n ']
t_f = np.array([train_freq.get(c,0) for c in plot_chars], dtype=float)
g_f = np.array([gen_freq.get(c,0)   for c in plot_chars], dtype=float)
t_f /= t_f.sum(); g_f /= g_f.sum()

x = np.arange(len(plot_chars))
fig, ax = plt.subplots(figsize=(15,4))
ax.bar(x-0.2, t_f, 0.4, label='Training data', color='steelblue', alpha=0.8)
ax.bar(x+0.2, g_f, 0.4, label='Generated text', color='coral',    alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(['\\n' if c=='\n' else 'SPC' if c==' ' else c
                     for c in plot_chars], fontsize=7, rotation=45)
ax.set_ylabel('Relative frequency')
ax.set_title('Character Frequency: Training Data vs Generated Text\n'
             '(closer match = better model)', fontsize=12)
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

---
## 📌 Section 11 — Complete Equation Reference

```
═══════════════════════════════════════════════════════════════
CHARACTER-LEVEL RNN — COMPLETE EQUATIONS
═══════════════════════════════════════════════════════════════

INPUT ENCODING
  x_t = one_hot(char_t)                       shape: (V,)

FORWARD PASS  (repeated at each time step t)
  z_t = W_xh · x_t  +  W_hh · h_{t-1}  +  b_h
  h_t = tanh(z_t)                             hidden state  shape:(H,)
  y_t = W_hy · h_t  +  b_y                   logits        shape:(V,)

OUTPUT
  p_t = softmax(y_t)        p_t[i] = P(char_i | context)

LOSS  (cross-entropy, summed)
  L_t = -log( p_t[correct_char_t] )
  L   = Σ_{t=1}^{T}  L_t

BPTT
  ∂L/∂W_hh  =  Σ_t  ∂L_t/∂W_hh
  ∂h_t/∂h_{t-1}  =  tanh'(z_t) · W_hh = (1 − h_t²) · W_hh
  |tanh'| ≤ 1  →  vanishes exponentially over many steps

GENERATION  (temperature T)
  p_t   = softmax(y_t / T)
  char_{t+1} ~ Categorical(p_t)
  x_{t+1} = one_hot(char_{t+1})   ← fed back as next input

LSTM CELL  (avoids vanishing gradients)
  f_t = σ(W_f · [h_{t-1}, x_t] + b_f)   ← forget gate
  i_t = σ(W_i · [h_{t-1}, x_t] + b_i)   ← input gate
  C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C) ← candidate
  C_t = f_t ⊗ C_{t-1}  +  i_t ⊗ C̃_t     ← ADDITIVE update!
  o_t = σ(W_o · [h_{t-1}, x_t] + b_o)   ← output gate
  h_t = o_t ⊗ tanh(C_t)

PARAMETER COUNT  (V=vocab, H=hidden)
  W_xh: (H×V)   W_hh: (H×H)   W_hy: (V×H)
  Total = H·V + H² + V·H + H + V
  Example: H=256, V=65  →  ~100K params
═══════════════════════════════════════════════════════════════
```

---
## 📚 References

| Resource | Link |
|----------|------|
| Karpathy's blog (primary source) | https://karpathy.github.io/2015/05/21/rnn-effectiveness/ |
| Minimal char-RNN in 100 lines (NumPy) | https://gist.github.com/karpathy/d4dee566867f8291f086 |
| Full char-rnn repo | https://github.com/karpathy/char-rnn |
| Understanding LSTMs (Colah's blog) | https://colah.github.io/posts/2015-08-Understanding-LSTMs/ |
| nanoGPT — modern successor | https://github.com/karpathy/nanoGPT |

---
*Notebook based on the CharRNN handout — Andrej Karpathy, "The Unreasonable Effectiveness of RNNs" (2015)*